# Part 1: Getting started

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np

%matplotlib inline
# set the random seed for reproducibility
seed = 'uzh2025'
seed = int.from_bytes(seed.encode('utf-8')) % 2**32
np.random.seed(seed)
import tensorflow as tf

tf.random.set_seed(seed)

## Fetch the jet tagging dataset from Open ML

**Note**: this takes about a minute to download the dataset

In [ ]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

### Let's print some information about the dataset
Print the feature names and the dataset shape

In [ ]:
print(f"Feature names: {data['feature_names']}\n")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}\n")
print(f"First 10 samples, X:\n {X[:10]}\n")
print(f"First 10 samples, y:\n {y[:10]}\n")

As you saw above, the `y` target is an array of strings of the jet flavour, e.g. \['g', 'w',...\] etc.
We need to make this a "One Hot" encoding for the training.
Then, split the dataset into training and validation sets

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)
y = to_categorical(y, 5)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(y[:5])

In [ ]:
scaler = StandardScaler()
X_train_val = scaler.fit_transform(X_train_val)
X_test = scaler.transform(X_test)

Save the preprocessed and train-test-split arrays to files so that we can skip the steps above next time.

In [ ]:
np.save('X_train_val.npy', X_train_val)
np.save('X_test.npy', X_test)
np.save('y_train_val.npy', y_train_val)
np.save('y_test.npy', y_test)
np.save('classes.npy', le.classes_)

## Now construct a model
We'll use 3 hidden layers with 64, then 32, then 32 neurons. Each layer will use `relu` activation.
Add an output layer with 5 neurons (one for each class), then finish with Softmax activation.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from callbacks import all_callbacks

In [ ]:
model = Sequential()
model.add(Dense(64, input_shape=(16,), name='fc1', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu1'))
model.add(Dense(32, name='fc2', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu2'))
model.add(Dense(32, name='fc3', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu3'))
model.add(Dense(5, name='output', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='softmax', name='softmax'))

## Train the model
We'll use Adam optimizer with categorical crossentropy loss.
The callbacks will decay the learning rate and save the model into a directory 'model_1'
The model isn't very complex, so this should just take a few minutes even on the CPU.
If you've restarted the notebook kernel after training once, set `train = False` to load the trained model.

In [ ]:
train = True
if train:
    adam = Adam(lr=0.0001)
    model.compile(optimizer=adam, loss=['categorical_crossentropy'], metrics=['accuracy'])
    callbacks = all_callbacks(
        stop_patience=1000,
        lr_factor=0.5,
        lr_patience=10,
        lr_epsilon=0.000001,
        lr_cooldown=2,
        lr_minimum=0.0000001,
        outputDir='model_1',
    )
    model.fit(
        X_train_val,
        y_train_val,
        batch_size=1024,
        epochs=10,
        validation_split=0.25,
        shuffle=True,
        callbacks=callbacks.callbacks,
    )
else:
    from tensorflow.keras.models import load_model

    model = load_model('model_1/KERAS_check_best_model.h5')

## Check performance
Check the accuracy and make a ROC curve

In [ ]:
import plotting
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

y_keras = model.predict(X_test)
print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)

# Quantization Aware Training

Now we train a new model using Quantization Aware Training via `QKeras`.

We replace layers like `Dense` and `Activation` with `QDense` and `QActivation`, and specifiy quantizers to use for the weights, biases and activations.

Notice that the model is still a `Keras` model, and the last layer is a regular `Keras` `Softmax` layer. In general you can mix-and-match normal `Keras` layers with `QKeras` ones.

In [ ]:
from qkeras.qlayers import QDense, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

In [ ]:
qmodel = Sequential()
qmodel.add(
    QDense(
        64,
        input_shape=(16,),
        name='fc1',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
    )
)
qmodel.add(QActivation(activation=quantized_relu(6), name='relu1'))
qmodel.add(
    QDense(
        32,
        name='fc2',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
    )
)
qmodel.add(QActivation(activation=quantized_relu(6), name='relu2'))
qmodel.add(
    QDense(
        32,
        name='fc3',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
    )
)
qmodel.add(QActivation(activation=quantized_relu(6), name='relu3'))
qmodel.add(
    QDense(
        5,
        name='output',
        kernel_quantizer=quantized_bits(6, 0, alpha=1),
        bias_quantizer=quantized_bits(6, 0, alpha=1),
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
    )
)
qmodel.add(Activation(activation='softmax', name='softmax'))

In [ ]:
train = True
if train:
    adam = Adam(lr=0.0001)
    qmodel.compile(optimizer=adam, loss=['categorical_crossentropy'], metrics=['accuracy'])
    callbacks = all_callbacks(
        stop_patience=1000,
        lr_factor=0.5,
        lr_patience=10,
        lr_epsilon=0.000001,
        lr_cooldown=2,
        lr_minimum=0.0000001,
        outputDir='qmodel_1',
    )
    qmodel.fit(
        X_train_val,
        y_train_val,
        batch_size=1024,
        epochs=30,
        validation_split=0.25,
        shuffle=True,
        callbacks=callbacks.callbacks,
    )
    qmodel.save('qmodel_1/KERAS_check_best_model.h5')
else:
    from tensorflow.keras.models import load_model
    from qkeras.utils import _add_supported_quantized_objects

    co = {}
    _add_supported_quantized_objects(co)
    qmodel = load_model('qmodel_1/KERAS_check_best_model.h5', custom_objects=co)

## Compare

Plot the ROC curve of the QKeras model alongside the Keras model we trained earlier, and print the accuracy of both. If we chose appropriate quantization, they should perform quite similarly 🤞

In [ ]:
y_qkeras = qmodel.predict(X_test)

print("Accuracy Keras:  {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("Accuracy QKeras: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_qkeras, axis=1))))

plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_qkeras, le.classes_, linestyle='--')

# Convert to FPGA code using `hls4ml`

Now we use `hls4ml` to transpile our trained Neural Network model into High Level Synthesis code that can by synthesized for FPGA. We first of all produce a configuration that details all of the settings to use in the FPGA code generation, including quantization and other parameters. When using QKeras, the quantization settings can be extracted from the `qmodel` that we provide.

After producing and editing the configuration, we `convert` the model into an `hls4ml` model object. This captures all of the parameters of the neural network, and provides an interface to produce FPGA code products for the model. We start by writing the project files, which you will find under `qmodel_q/hls4ml_prj`. This includes the C++ (HLS) description of the model we provided, including the architecture, its weights and bias values, the quantization settings, and the optimized inference functions.

In [ ]:
import hls4ml

In [ ]:
config = hls4ml.utils.config_from_keras_model(qmodel, granularity='name', backend='Vitis', default_precision='fixed<16,7>')
config['LayerName']['softmax']['exp_table_t'] = 'ap_fixed<18,8>'
config['LayerName']['softmax']['inv_table_t'] = 'ap_fixed<18,4>'
config['LayerName']['softmax']['Implementation'] = 'latency'
print("-----------------------------------")
plotting.print_dict(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel, hls_config=config, backend='Vitis', output_dir='qmodel_1/hls4ml_prj', part='xcu250-figd2104-2L-e'
)
hls_model.write()

## Validate

The FPGA design flow is time consuming, and working with FPGA hardware is not trivial. It's always good practice to check on our CPU that the FPGA design is correct before we go further. Since our Neural Network has been converted to C++ code (HLS), we can compile and run it on the CPU as well. `hls4ml` provides bindings and scripts to compile the C++ code and interact with it from Python. Here we compile the code and make an inference (on CPU) on our `X_test` data. This emulation produces bit-accurate results to what we would see on the FPGA device, given the same inputs.

In [ ]:
hls_model.compile()
y_hls4ml = hls_model.predict(np.ascontiguousarray(X_test))

## Compare

Let's again produce the ROC curve, now with the original Keras model, the quantized QKeras model, and the hls4ml emulation of the QKeras model.

In [ ]:
print("Accuracy Keras:  {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("Accuracy QKeras: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_qkeras, axis=1))))
print("Accuracy hls4ml: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls4ml, axis=1))))


plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, [c + ' Keras' for c in le.classes_])
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_qkeras, [c + ' QKeras' for c in le.classes_], linestyle='--')
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_hls4ml, [c + ' hls4ml' for c in le.classes_], linestyle=':')

# Synthesize

Everything we've done so far has taken place on the CPU. In order to go further with the FPGA we need to _synthesize_ the HLS C++ code into a lower level: _Harwdare Description Language_. This process uses software from the FPGA vendor, in this case AMD Xilinx, called `Vitis HLS`.

This might not be available in the environment you're using, so if it's available then the cell below will run this step, otherwise this step will pick up a report that's provided. The report output summarises estimates of the latency and resource usage of the neural network. Look at the absolute inference latency, and the percentage total resource usage of each resource type.

This report is a so called "Out of Context" report - meaning that the Neural Network is not interfaced to any other component. In a real system the Neural Network needs to be connected to the outside world somehow to receive its inputs and do something with its outputs. In Level 1 Trigger hardware, this can be achieved with extremely low latency and high throughput, but in other edge hardware that might not be the case. This ultra low latency inference can only be obtained with a correspondingly high performance I/O.

In [ ]:
import shutil
if shutil.which('vitis_hls') is not None:
    # Vitis HLS is available, run csynthesis
    hls_model.build()
    qreport = 'qmodel_1/hls4ml_prj'
else:
    # use an already made report
    qreport = 'reports/qmodel_1/hls4ml_prj'

In [ ]:
hls4ml.report.read_vivado_report(qreport)

## Compare

We can also synthesize the first, non-quantized Keras model that we trained. We use a method called "Post Training Quantization" where the floating point weights, biases, and activations are mapped onto fixed point representations after the model is trained. This usually comes with some performace degradation if the bitwidths are not wide enough, so `hls4ml` defaults to using 16 bits for everything. Compare this reports to see how much more efficient Quantization Aware Training can be!

In [ ]:
if shutil.which('vitis_hls') is not None:
    fconfig = hls4ml.utils.config_from_keras_model(model, granularity='model', backend='Vitis')
    print("-----------------------------------")
    print("Configuration")
    plotting.print_dict(fconfig)
    print("-----------------------------------")
    hls_fmodel = hls4ml.converters.convert_from_keras_model(
        model, hls_config=fconfig, backend='Vitis', output_dir='model_1/hls4ml_prj', part='xcu250-figd2104-2L-e'
    )
    hls_fmodel.build()
    freport = 'model_1/hls4ml_prj'
else:
    freport = 'reports/model_1/hls4ml_prj'

In [ ]:
hls4ml.report.read_vivado_report(freport)